# Phase 1 — Data pipeline (Multi30k En→De)

**Goal:** turn raw sentence pairs into batched tensors the model can eat — and *see*
every transformation so there are no black boxes:

1. Load Multi30k (with a tiny offline fallback so this notebook always runs).
2. Train a **joint** BPE tokenizer (shared En+De vocab, ~10k).
3. `TranslationDataset` → `(src_ids, tgt_ids)` with `<bos>`/`<eos>`.
4. `collate_fn` → pad to max-in-batch, build the **src pad mask** and the
   **tgt mask = pad × causal** (additive `-inf` form, the shape the model expects).

**Feel-checks:** string→ids→string roundtrip, a raw padded batch, the causal-mask
heatmap, and a sentence-length histogram to pick `max_len`.

In [ ]:
# Bootstrap: put the repo root on sys.path so `import transformer` works from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

## 1. Load the dataset

We try HuggingFace `datasets` first. If there's no network, we fall back to a tiny
hardcoded En–De sample so the rest of the notebook still runs end-to-end (just on
fewer examples). The shape of the data is identical either way: a list of
`{"en": ..., "de": ...}` dicts per split.

In [ ]:
TINY_FALLBACK = [
    ("A man in a blue shirt is standing on a ladder cleaning a window.",
     "Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster."),
    ("A group of people are sitting outside a building.",
     "Eine Gruppe von Menschen sitzt vor einem Gebäude."),
    ("Two young men are smiling and laughing in a park.",
     "Zwei junge Männer lächeln und lachen in einem Park."),
    ("A little girl is running across a grassy field.",
     "Ein kleines Mädchen rennt über eine Wiese."),
    ("A dog jumps to catch a red ball in the air.",
     "Ein Hund springt, um einen roten Ball in der Luft zu fangen."),
    ("Three children are playing soccer on the beach.",
     "Drei Kinder spielen Fußball am Strand."),
    ("An old woman is selling vegetables at the market.",
     "Eine alte Frau verkauft Gemüse auf dem Markt."),
    ("A chef in a white hat is preparing food in a kitchen.",
     "Ein Koch mit weißer Mütze bereitet Essen in einer Küche zu."),
]

def load_multi30k():
    """Return dict(split -> list of (en, de)). Falls back to TINY_FALLBACK if offline."""
    try:
        from datasets import load_dataset
        ds = load_dataset("bentrevett/multi30k")
        out = {}
        for split in ("train", "validation", "test"):
            rows = ds[split]
            out[split] = list(zip(rows["en"], rows["de"]))
        print("loaded Multi30k from HuggingFace:",
              {k: len(v) for k, v in out.items()})
        return out, True
    except Exception as e:
        print(f"[offline fallback] could not load from HF ({type(e).__name__}: {e})")
        # Replicate the tiny sample so splits aren't empty.
        data = TINY_FALLBACK * 8
        out = {"train": data, "validation": TINY_FALLBACK[:4], "test": TINY_FALLBACK[:4]}
        print("using TINY_FALLBACK:", {k: len(v) for k, v in out.items()})
        return out, False

data, online = load_multi30k()
train_pairs = data["train"]

## 2. Eyeball the raw pairs

Before any tokenization, just read a few. Translation data is messy — casing,
punctuation, compounding in German. Get a feel for what the model has to learn.

In [ ]:
for en, de in train_pairs[:5]:
    print("EN:", en)
    print("DE:", de)
    print()

## 3. Train a joint BPE tokenizer

A **joint** (shared) vocabulary means English and German share one token table. This
lets weight tying (one embedding for src/tgt + output projection) work and helps with
shared subwords (names, punctuation, cognates). Special tokens go first so their ids
are stable and small: `<pad>=0, <bos>=1, <eos>=2, <unk>=3`.

We train on the concatenation of all source and target sentences. Vocab size scales
down automatically for the tiny fallback corpus.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
SPECIALS = [PAD, BOS, EOS, UNK]
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

def train_joint_bpe(pairs, vocab_size=10000):
    corpus = [s for en, de in pairs for s in (en, de)]
    # Cap vocab for tiny corpora so the trainer doesn't choke.
    vocab_size = min(vocab_size, max(64, len(set(" ".join(corpus).split())) + len(SPECIALS)))
    tok = Tokenizer(models.BPE(unk_token=UNK))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIALS,
                                  show_progress=False)
    tok.train_from_iterator(corpus, trainer=trainer)
    return tok

tokenizer = train_joint_bpe(train_pairs, vocab_size=10000)
VOCAB = tokenizer.get_vocab_size()
print("vocab size:", VOCAB)
print("special ids:", {s: tokenizer.token_to_id(s) for s in SPECIALS})
assert [tokenizer.token_to_id(s) for s in SPECIALS] == [0, 1, 2, 3], "special ids must be 0..3"

## Feel-check: string → ids → string roundtrip

Encode a few sentences, look at the subword pieces, then decode back. The pieces
won't be whole words — that's BPE working. Decoding should recover the original
(modulo whitespace normalization).

In [ ]:
for en, de in train_pairs[:5]:
    enc = tokenizer.encode(en)
    dec = tokenizer.decode(enc.ids)
    print("text  :", en)
    print("tokens:", enc.tokens)
    print("ids   :", enc.ids)
    print("decode:", dec)
    print()

## 4. `TranslationDataset`

Each item is a `(src_ids, tgt_ids)` pair. We wrap the **target** with `<bos>`/`<eos>`
so teacher forcing has a start token and a stop signal. (The source needs no bos/eos
for this task — the encoder sees the whole sentence at once; we add `<eos>` to src too,
a common, harmless convention that gives the encoder an explicit end marker.)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TranslationDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=64):
        self.pairs = pairs
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def _encode(self, text, add_bos):
        ids = self.tok.encode(text).ids[: self.max_len - 2]
        body = ([BOS_ID] if add_bos else []) + ids + [EOS_ID]
        return torch.tensor(body, dtype=torch.long)

    def __getitem__(self, i):
        en, de = self.pairs[i]
        src = self._encode(en, add_bos=False)   # src: tokens + <eos>
        tgt = self._encode(de, add_bos=True)     # tgt: <bos> + tokens + <eos>
        return src, tgt

train_ds = TranslationDataset(train_pairs, tokenizer, max_len=64)
src0, tgt0 = train_ds[0]
print("src0:", src0.tolist())
print("tgt0:", tgt0.tolist())
print("decoded src:", tokenizer.decode(src0.tolist()))
print("decoded tgt:", tokenizer.decode(tgt0.tolist()))

### Why the teacher-forcing shift?

During training the decoder receives the **ground-truth** target prefix, not its own
predictions. This is *teacher forcing*. The `collate_fn` implements it with a simple
one-position shift:

```
tgt_in  : <bos>  A    man  walks  …   (fed to the decoder as input)
tgt_out :  A    man  walks  …   <eos> (what the model must predict at each step)
```

At step *t* the decoder sees `tgt_in[0..t]` and must predict `tgt_out[t]`.
Using the gold prefix (rather than the model's own possibly-wrong output) keeps
training stable — a mistake at step 2 won't corrupt every subsequent step.

The trade-off is **exposure bias**: at inference the model is fed its own outputs,
not gold tokens, so small errors can compound. Scheduled sampling can mitigate this,
but pure teacher forcing is standard for this architecture.


## 5. `collate_fn` — padding + masks

This is the subtle part. For a batch we:

- pad `src` and `tgt` to the longest in the batch with `<pad>=0`;
- split target into `tgt_in` (all but last) and `tgt_out` (all but first) — the
  teacher-forcing shift, so position *t* predicts token *t+1*;
- build **boolean** pad masks (`True` = is-pad = blocked).

We return boolean masks here and let the model convert them to additive `-inf` form
(that's how `transformer/model.py` is wired — masks are additive, shape
`(B, 1, Tq, Tk)`). Keeping masks boolean at the data layer keeps this notebook backend-
agnostic and easy to visualize.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src = pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)   # (B, S)
    tgt = pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)   # (B, T+1)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]                          # teacher-forcing shift
    src_pad = src.eq(PAD_ID)        # (B, S)  True where padded
    tgt_pad = tgt_in.eq(PAD_ID)     # (B, T)
    return {
        "src": src, "tgt_in": tgt_in, "tgt_out": tgt_out,
        "src_pad": src_pad, "tgt_pad": tgt_pad,
    }

loader = DataLoader(train_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
batch = next(iter(loader))
for k, v in batch.items():
    print(f"{k:8s}: {tuple(v.shape)}  dtype={v.dtype}")

## Feel-check: look at one raw padded batch

Print the source ids of a batch as a grid. You should see real token ids on the left
and a block of `0`s (pad) filling each row out to the longest sentence. That ragged-
right zero pattern is exactly what the pad mask will block.

In [ ]:
print("src batch (rows = sentences, 0 = <pad>):")
for row in batch["src"]:
    print(" ", row.tolist())
print()
print("src_pad mask (True = blocked):")
for row in batch["src_pad"]:
    print(" ", row.tolist())

## Feel-check: visualize the causal mask

The decoder must not see the future. Position *t* may attend to positions `0..t` only,
so the allowed pattern is **lower-triangular**. Here we build the same boolean causal
mask the model uses and plot it: bright = allowed, dark = blocked.

In [ ]:
import matplotlib.pyplot as plt

T = batch["tgt_in"].size(1)
causal_blocked = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # True = blocked
allowed = (~causal_blocked).float()

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(allowed, cmap="gray", vmin=0, vmax=1)
ax.set_title(f"Causal mask (T={T})\nwhite = can attend, black = blocked (future)")
ax.set_xlabel("key position (attended to)")
ax.set_ylabel("query position (doing the attending)")
plt.tight_layout(); plt.show()
print("Row t can attend to columns 0..t — that's why it's lower-triangular.")

### Combined decoder mask — causal AND padding together

The decoder's self-attention blocks two things simultaneously:
- **Causal** — query at position *t* must not attend to any key at position *> t*.
- **Key-padding** — no query should attend to a `<pad>` key (those aren't real tokens).

Visualising them together makes it obvious which positions are reachable (white) vs.
completely blocked (black) for every query/key pair in the batch.


In [ ]:
# Key-padding mask: column j blocked for ALL queries if tgt_in[j] is <pad>
key_pad_blocked = batch["tgt_pad"][0].unsqueeze(0).expand(T, T)  # (T, T)

combined_blocked = causal_blocked | key_pad_blocked

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
data = [causal_blocked.float(), key_pad_blocked.float(), combined_blocked.float()]
titles = [
    "Causal mask\n(future blocked)",
    "Key-padding mask\n(pad tokens blocked)",
    "Combined target mask\n(causal ∪ padding)",
]
for ax, mat, title in zip(axes, data, titles):
    ax.imshow(1 - mat, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("key position"); ax.set_ylabel("query position")
plt.suptitle("Decoder self-attention masks  (white = can attend, black = blocked)",
             y=1.02, fontsize=10)
plt.tight_layout(); plt.show()
print(f"Blocked positions: causal={int(causal_blocked.sum())} "
      f"| pad={int(key_pad_blocked.sum())} "
      f"| combined={int(combined_blocked.sum())}")


## Feel-check: sentence-length histogram → pick `max_len`

Tokenized lengths tell us where to set `max_len`. Pick a value that covers the vast
majority of sentences without wasting compute on rare long ones (attention cost is
quadratic in length).

In [ ]:
import numpy as np

src_lens = [len(tokenizer.encode(en).ids) for en, _ in train_pairs]
tgt_lens = [len(tokenizer.encode(de).ids) for _, de in train_pairs]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(src_lens, bins=30, alpha=0.6, label="EN (src)")
ax.hist(tgt_lens, bins=30, alpha=0.6, label="DE (tgt)")
ax.axvline(np.percentile(src_lens + tgt_lens, 99), color="k", ls="--",
           label="99th pct")
ax.set_xlabel("tokens"); ax.set_ylabel("count"); ax.legend()
ax.set_title("Tokenized sentence lengths")
plt.tight_layout(); plt.show()

for p in (50, 90, 95, 99, 100):
    print(f"  {p:3d}th pct: src={np.percentile(src_lens, p):.0f}  "
          f"tgt={np.percentile(tgt_lens, p):.0f}")
print("\n-> a max_len around the 99th percentile (+2 for bos/eos) is a sensible start.")

## Done when

- The dataloader yields batches with correct shapes and the right masks.
- You've *seen* the roundtrip, the padding pattern, the causal triangle, and the
  length distribution.

**Next:** once you've run this and it feels right, we lift the proven pieces
(`tokenizer.py`, `data.py`) into the `transformer/` package, then move to
`02_components.ipynb` to build positional encoding, MHA, FFN, and the enc/dec layers
from scratch — each with its own shape test and feel-check.